In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [3]:
from waymo_agent.notebook_imports import *

In [4]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *
from waymo_agent.action_heuristic.heuristic_simple import PricingAgent, DispatchAgent, RepositionAgent
from waymo_agent.models.ppo_model import RideShareActorCritic
from waymo_agent.graph_env.ENV import RideShareEnv
from waymo_agent.models.ppo_model import *

In [5]:
from waymo_agent.models.ppo_model import *
from waymo_agent.models.evaluate import *
from waymo_agent.models.train_utils import *
from waymo_agent.models.torch_np_utils import *
from waymo_agent.models.torch_np_utils import _flat_obs

In [6]:
from kret_sandbox.VIS import dtt
from kret_sandbox.exp_decay import exp_decay_half_life, get_gamma_from_half_life

In [7]:
def get_obs_tuple(env: RideShareEnv):
    veh = env.observation_curr["vehicles"]
    req = env.observation_curr["pending_requests"]
    rides = env.observation_curr["active_rides"]
    return veh, req, rides


def get_obs_tuple_from_obs(obs: ObservationDict):
    veh = obs["vehicles"]
    req = obs["pending_requests"]
    rides = obs["active_rides"]
    return veh, req, rides

## Env Init

In [8]:
env_cfg = EnvConfig(
    max_episode_steps=60 * 1,
    vehicle_per_node=0.10,
    lambda_per_node=0.05,
    max_pending_requests=100,
    max_new_requests_per_step=20,
)
plt_cfg = PlotConfig()
env_model = RideShareEnv(env_cfg, plt_cfg)
env = env_model

Assigned lambda values to nodes. Total lambda: 39.1000 (target: 39.1000)


In [9]:
env_model.config.no_action_id

-1

In [10]:
obs, info = env_model.reset()

In [11]:
veh, req, rides = get_obs_tuple_from_obs(obs)

In [12]:
veh_enr, req_enr, rides_enr = get_obs_tuple(env_model)

In [13]:
veh_enr.head(4)

,vehicle_id,loc_x_norm,loc_y_norm,battery,status,ride_id
0,0,-0.562126,-0.741366,0.993055,0,-1
1,1,-0.143045,-0.502034,0.752366,0,-1
2,2,-0.234294,0.170498,0.687362,0,-1
3,3,0.095884,-0.175229,0.891168,0,-1


# Test Subheads


In [14]:
GAMMA = get_gamma_from_half_life(env_cfg.max_episode_steps // 2)
round(GAMMA, 5)

0.97716

## Pricing Head

In [15]:
obs_np, _ = env_model.reset(seed=0)
obs_t = obs_pd_to_torch(obs_np)  # dict[str, Tensor] on DEVICE (if your converter does that)

# --- build encoder + head (standalone) ---
# Use same obs_dim calculation RideShareActorCritic uses:
x = _flat_obs(obs_t)  # (obs_dim,)
obs_dim = int(x.numel())

hidden = 256
enc = SharedEncoder(obs_dim, hidden).to(DEVICE).eval()
head = PricingHead(hidden, env_model.config).to(DEVICE).eval()

with torch.no_grad():
    h = enc(x)  # (hidden,)
    # PricingHead expects h shaped like (hidden,) or (B, hidden). We'll enforce (B, hidden).
    if h.ndim == 1:
        hB = h.unsqueeze(0)  # (1, hidden)
    else:
        hB = h

    # 1) sample action
    a = head.act(hB, obs=obs_t, deterministic=False)  # (1, max_pending)
    assert a.shape[-1] == env_model.config.max_pending_requests
    assert torch.isfinite(a).all()
    assert (a > 0).all()

    # 2) logprob/entropy should be finite
    logp, ent = head.log_prob_and_entropy(hB, a, obs=obs_t)
    assert logp.shape == (1,)
    assert ent.shape == (1,)
    assert torch.isfinite(logp).all()
    assert torch.isfinite(ent).all()

    # 3) mask behavior: where pricing_mask==0, action should be eps_price
    if "pricing_mask" in obs_t:
        pm = obs_t["pricing_mask"].to(device=a.device, dtype=a.dtype)  # (50,) likely
        # broadcast pm to (1, 50)
        pmB = pm.unsqueeze(0) if pm.ndim == 1 else pm
        eps = head.eps_price
        masked = pmB <= 0.0
        if masked.any():
            assert torch.allclose(
                a[masked], torch.full_like(a[masked], eps)
            ), f"Masked prices not set to eps_price={eps}"

print("PricingHead smoke test: PASS ✅")

PricingHead smoke test: PASS ✅


## Reposition Head

In [16]:
obs_np, _ = env.reset(seed=0)
obs_t = obs_pd_to_torch(obs_np)  # dict[str, Tensor] on DEVICE

# --- Build encoder + head (standalone) ---
x = _flat_obs(obs_t)  # (obs_dim,)
obs_dim = int(x.numel())

hidden = 256
enc = SharedEncoder(obs_dim, hidden).to(DEVICE).eval()
head = RepositionHead(hidden, env.num_vehicles).to(DEVICE).eval()

with torch.no_grad():
    h = enc(x)  # (hidden,)
    hB = h.unsqueeze(0)  # (1, hidden)

    # 1) sample action
    a = head.act(hB, obs=obs_t, deterministic=False)  # (1, num_veh, 2)
    assert a.shape == (1, env.num_vehicles, 2), a.shape
    assert torch.isfinite(a).all()
    assert (a >= -1.0).all() and (a <= 1.0).all()

    # 2) logprob/entropy finite + shapes
    logp, ent = head.log_prob_and_entropy(hB, a, obs=obs_t)
    assert logp.shape == (1,), logp.shape
    assert ent.shape == (1,), ent.shape
    assert torch.isfinite(logp).all()
    assert torch.isfinite(ent).all()

    # 3) mask behavior:
    # make a fake mask where only first half vehicles are "idle"
    m = obs_t["dispatch_mask"].clone()
    m[:] = 0.0
    m[: env.num_vehicles // 2] = 1.0

    obs_t_masked = dict(obs_t)
    obs_t_masked["dispatch_mask"] = m

    a2 = head.act(hB, obs=obs_t_masked, deterministic=False)  # (1, num_veh, 2)
    # vehicles where mask==0 should be exactly 0 (per your implementation)
    mB = m.unsqueeze(0).to(device=a2.device, dtype=a2.dtype)  # (1, num_veh)
    # On masked vehicles, a2 should be ~0 for both coords
    assert torch.allclose(a2 * (1.0 - mB.unsqueeze(-1)), torch.zeros_like(a2)), "Masked reposition actions not zeroed"

    # logprob should still be finite under mask
    logp2, ent2 = head.log_prob_and_entropy(hB, a2, obs=obs_t_masked)
    assert torch.isfinite(logp2).all()
    assert torch.isfinite(ent2).all()

print("RepositionHead smoke test: PASS ✅")

RepositionHead smoke test: PASS ✅


## Dispatch Head

In [17]:
obs_np, _ = env.reset(seed=0)
obs_t = obs_pd_to_torch(obs_np)

num_veh = env.num_vehicles
max_pending = env.config.max_pending_requests

# ------------------------------------------------------------
# Build encoder + DispatchHead (standalone)
# ------------------------------------------------------------
x = _flat_obs(obs_t)  # (obs_dim,)
obs_dim = int(x.numel())

hidden = 256
enc = SharedEncoder(obs_dim, hidden).to(DEVICE).eval()
head = DispatchHead(hidden, max_pending, num_veh).to(DEVICE).eval()

with torch.no_grad():
    # ensure batch dimension
    h = enc(x.unsqueeze(0))  # (1, hidden)

    # --------------------------------------------------------
    # 1) Sample stochastic dispatch action
    # --------------------------------------------------------
    a = head.act(h, obs=obs_t, deterministic=False)
    assert a.shape == (1, max_pending), a.shape
    assert a.dtype in (torch.int64, torch.int32)
    assert torch.isfinite(a).all()

    # dispatch values must be in [-1 .. num_veh-1]
    assert (a >= -1).all()
    assert (a < num_veh).all()

    # --------------------------------------------------------
    # 2) log_prob / entropy finite
    # --------------------------------------------------------
    logp, ent = head.log_prob_and_entropy(h, a, obs=obs_t)
    assert logp.shape == (1,)
    assert ent.shape == (1,)
    assert torch.isfinite(logp).all()
    assert torch.isfinite(ent).all()

    # --------------------------------------------------------
    # 3) Mask semantics: forbid all vehicles
    # --------------------------------------------------------
    obs_t_masked = dict(obs_t)
    obs_t_masked["dispatch_mask"] = torch.zeros_like(obs_t["dispatch_mask"])

    a_masked = head.act(h, obs=obs_t_masked, deterministic=False)
    # All dispatches must be NO-ACTION (-1)
    assert (a_masked == -1).all(), "DispatchHead failed mask semantics: non-idle vehicle assigned"

    logp_m, ent_m = head.log_prob_and_entropy(h, a_masked, obs=obs_t_masked)
    assert torch.isfinite(logp_m).all()
    assert torch.isfinite(ent_m).all()

    # --------------------------------------------------------
    # 4) Partial mask: only allow first half vehicles
    # --------------------------------------------------------
    m = torch.zeros_like(obs_t["dispatch_mask"])
    m[: num_veh // 2] = 1.0
    obs_t_half = dict(obs_t)
    obs_t_half["dispatch_mask"] = m

    a_half = head.act(h, obs=obs_t_half, deterministic=False)
    a_np = a_half.squeeze(0).cpu().numpy()

    # Any actual vehicle assignment must be within allowed range
    for d in a_np:
        if d >= 0:
            assert d < num_veh // 2, f"Assigned masked vehicle {d} with dispatch_mask=0"

    # --------------------------------------------------------
    # 5) Deterministic policy sanity
    # --------------------------------------------------------
    a_det = head.act(h, obs=obs_t, deterministic=True)
    assert a_det.shape == (1, max_pending)
    assert torch.isfinite(a_det).all()

print("DispatchHead smoke test: PASS ✅")

DispatchHead smoke test: PASS ✅


In [18]:
# ----------------------------
# Helper: build a valid "mostly no-op" action
# ----------------------------
def make_action(
    env: RideShareEnv,
    *,
    prices: np.ndarray | None = None,
    dispatch: np.ndarray | None = None,
    reposition: np.ndarray | None = None,
):
    if prices is None:
        prices = np.zeros(env.config.max_pending_requests, dtype=np.float64)
    if dispatch is None:
        dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
    if reposition is None:
        reposition = np.zeros((env.num_vehicles, 2), dtype=np.float32)
    return {"prices": prices, "dispatch": dispatch, "reposition": reposition}


# ----------------------------
# Test: dispatch creates assignment when feasible
# ----------------------------
obs, _ = env.reset(seed=0)

accepted_req_idx = None
idle_veh_idx = None

# 1) Drive the env until we see at least one ACCEPTED request (pricing only, no dispatch)
for t in range(200):
    a = make_action(env, prices=np.zeros(env.config.max_pending_requests, dtype=np.float64))
    obs, r, term, trunc, info = env.step(a)

    req = env.observation_curr["pending_requests"]
    # accepted mask over rows
    f_acc = (req["status"] == RequestStatusEnum.ACCEPTED.value).to_numpy()
    if f_acc.any():
        accepted_req_idx = int(np.argmax(f_acc))
        break
    if term or trunc:
        break

assert accepted_req_idx is not None, "Never observed an ACCEPTED request — pricing/arrival dynamics may be broken."

# 2) Pick an IDLE vehicle from dispatch_mask (this is the *row index* space!)
veh = env.observation_curr["vehicles"]
dispatch_mask = env.observation_curr["dispatch_mask"]  # bool array
idle_idxs = np.where(dispatch_mask)[0]
assert idle_idxs.size > 0, "No IDLE vehicles available; cannot test dispatch."

idle_veh_idx = int(idle_idxs[0])

# Snapshot pre-state for the chosen row indices
pre_req_status = int(env.observation_curr["pending_requests"].iloc[accepted_req_idx]["status"])
pre_veh_status = int(env.observation_curr["vehicles"].iloc[idle_veh_idx]["status"])
pre_veh_rideid = int(env.observation_curr["vehicles"].iloc[idle_veh_idx]["ride_id"])

# Sanity: request is actually ACCEPTED, vehicle is actually IDLE
assert pre_req_status == RequestStatusEnum.ACCEPTED.value, (accepted_req_idx, pre_req_status)
assert pre_veh_status == VehicleStatusEnum.IDLE.value, (idle_veh_idx, pre_veh_status)

# 3) Dispatch that idle vehicle to that accepted request
dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
dispatch[accepted_req_idx] = idle_veh_idx

a2 = make_action(env, prices=np.zeros(env.config.max_pending_requests, dtype=np.float64), dispatch=dispatch)
obs2, r2, term2, trunc2, info2 = env.step(a2)

# 4) Validate post-state
req2 = env.observation_curr["pending_requests"]
veh2 = env.observation_curr["vehicles"]

post_req_status = int(req2.iloc[accepted_req_idx]["status"])
post_veh_status = int(veh2.iloc[idle_veh_idx]["status"])
post_veh_rideid = int(veh2.iloc[idle_veh_idx]["ride_id"])

assert post_req_status == RequestStatusEnum.ASSIGNED.value, (
    "Dispatch did not transition ACCEPTED -> ASSIGNED",
    accepted_req_idx,
    pre_req_status,
    post_req_status,
)

assert post_veh_status != VehicleStatusEnum.IDLE.value, (
    "Vehicle did not leave IDLE after a supposedly valid dispatch",
    idle_veh_idx,
    pre_veh_status,
    post_veh_status,
)

assert post_veh_rideid != pre_veh_rideid, (
    "Vehicle ride_id did not change after dispatch (expected new assignment)",
    idle_veh_idx,
    pre_veh_rideid,
    post_veh_rideid,
)

print("ENV dispatch feasibility test: PASS ✅")
print(f"  used accepted_req_idx={accepted_req_idx}, idle_veh_idx={idle_veh_idx}")
print(f"  req status: {pre_req_status} -> {post_req_status}")
print(f"  veh status: {pre_veh_status} -> {post_veh_status}, ride_id: {pre_veh_rideid} -> {post_veh_rideid}")

ValueError: Unable to coerce to Series, length must be 2: given 100

In [ ]:
for key, val in env.action_space.items():
    print(f"{key}: {val.shape=}")

dispatch: val.shape=(100,)
prices: val.shape=(100,)
reposition: val.shape=(50, 2)


In [ ]:
for key, val in env.observation_space.items():
    print(f"{key}: {val.shape=}")

active_rides: val.shape=(50, 9)
dispatch_mask: val.shape=(50,)
globals: val.shape=(5,)
pending_requests: val.shape=(100, 11)
pricing_mask: val.shape=(100,)
supply_demand_ratio: val.shape=(3,)
vehicles: val.shape=(50, 4)


In [ ]:
veh, req, rides = get_obs_tuple_from_obs(obs)

In [ ]:
req.f_need_dispatch.shape

(100,)